# MenuGraph — Visualisation

**Data:** `menu_graph_v1.json`. Same graph model as agentic implementation (`experiments.models.menu_graph.MenuGraph`).

- **Incoming edges** on a recipe → how often it is served.
- **Outgoing edges** from a meal period → variety (number of recipes offered).

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
backend = next((p for p in candidates if (p / "experiments" / "menu_graph_v1.json").exists()), Path.cwd() / "backend")
if str(backend) not in sys.path:
    sys.path.insert(0, str(backend))

from experiments.models.menu_graph import MenuGraph

path = backend / "experiments" / "menu_graph_v1.json"
graph = MenuGraph.from_json_path(path)

meta = graph.graph_metadata
print(f"Loaded: {meta.description} — {meta.cycle}")
print(f"Nodes: {len(graph.nodes)}  |  Edges: {len(graph.edges)}")
print(f"Recipe serve count (Hamburger): {graph.recipe_serve_count('M8958')}")
print(f"Period variety (Lunch): {graph.period_variety_count('Period_Lunch')}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for n in graph.nodes:
    G.add_node(n.id, node_type=n.type, label=n.name)
for e in graph.edges:
    G.add_edge(e.source, e.target)

type_color = {"Station": "#c0392b", "Day": "#2980b9", "MealPeriod": "#27ae60", "Recipe": "#e67e22"}
colors = [type_color.get(G.nodes[n].get("node_type", ""), "#95a5a6") for n in G.nodes]

pos = nx.spring_layout(G, k=1.2, seed=42, iterations=80)

plt.figure(figsize=(16, 12))
nx.draw_networkx_edges(G, pos, edge_color="#bdc3c7", arrows=True, arrowsize=10)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=300)
nx.draw_networkx_labels(G, pos, {n: G.nodes[n].get("label", n) for n in G.nodes}, font_size=6)
plt.title(meta.cycle)
plt.axis("off")
plt.tight_layout()
plt.show()

## Traversal: incoming / outgoing counts

`graph.count_incoming(node_id)`, `graph.count_outgoing(node_id)`, `graph.recipe_serve_count(recipe_id)`, `graph.period_variety_count(period_id)`.

In [ ]:
print("Recipe serve counts (incoming edges = how often served):")
for r in graph.get_recipes()[:8]:
    c = graph.recipe_serve_count(r.id)
    print(f"  {r.name}: {c}")
print("\nPeriod variety (outgoing edges = number of recipes offered):")
for p in graph.get_meal_periods():
    c = graph.period_variety_count(p.id)
    print(f"  {p.name}: {c}")